In [1]:
####### COMPILE ALL GRAPHS #######
import numpy as np
import pandas as pd
from scipy.stats import ttest_rel
import sys;sys.path.append("../gnn");sys.path.append("../networks")
import os
import json
import pickle
from utils import resolve_paths, load_atlas_data
#from gnnutils import *

READ_DATA_PATHS, WRITE_DATA_PATHS = resolve_paths(read_datasets=["Atlas Trade Data",
                                                                "Atlas Countries Data", 
                                                                "Atlas Products Data",
                                                                "BACI Data",
                                                                 "Graphs Data",
                                                                 "Results Data"], 
                                                write_datasets=["Graphs Data"])

Loading DATA_PATHS.yaml from ../../data/DATA_PATHS.yaml


In [2]:
def get_best_threshold(graph, model):
    """
    Auxiliary function to collect the threshold from the tuning phase, and compute the mean.
    """

    ts = []

    for seed in os.listdir(f"../gnn/models/training/No Ablation/{model}/{graph}"):
        for file in os.listdir(f"../gnn/models/training/No Ablation/{model}/{graph}/{seed}"):
            if file == "threshold.json":
                with open(f"../gnn/models/training/No Ablation/{model}/{graph}/{seed}/threshold.json", "r") as f:
                    data = json.load(f)["threshold"]
                    ts.append(data)

    return np.mean(ts)

## Settings

In [3]:
## Network settings
digits = 2 # 2 or 4
graph = "export" # "imports" or "total"
## Model
model = "GAT"  # Choose model: MLP, GCN, SAGE or GAT

#### Load data

In [4]:
# Read imports volume data
df = load_atlas_data(READ_DATA_PATHS["Atlas Trade Data"])
atlas = df.drop(columns=["coi", "eci", "pci"])

# Read Atlas product ID to commodity mapping
products = pd.read_csv(READ_DATA_PATHS["Atlas Products Data"], 
                       dtype={"code": str})
countries = pd.read_csv(READ_DATA_PATHS["Atlas Countries Data"], 
                        encoding="latin-1")
products = products[["product_id", "code", "name_short_en"]]
products["code"] = products["code"].str[:digits]
    
## Get commodity from mapping
atlas = atlas.merge(products, on="product_id", how="left", indicator=False)
atlas = atlas[~(atlas["code"].str.startswith("XX") & ~atlas["code"].str.startswith("99"))]
atlas_data = atlas.groupby(["year", "country_id", "partner_country_id", "code"]).sum().reset_index()

## Risk data
risk = pd.read_csv(READ_DATA_PATHS["Results Data"] +
                    f"/Risk Scores/{digits}_digits/{model}-{graph}-risk-scores.csv", 
                    dtype={"commodity": str}).rename(columns={"commodity": "code"})

# SRCA data
with open(READ_DATA_PATHS["Graphs Data"] + f"/{digits}_digits/SRCA.pickle", "rb") as f:
    rca_dict = pickle.load(f)

atlas_data.head()

,year,country_id,partner_country_id,code,product_id,export_value,import_value,name_short_en
0,2012,4,31,17,796,0.0,4300800.0,Sugarcane & sucrose
1,2012,4,31,20,819,0.0,34244.0,Fruit juices
2,2012,4,31,27,906,0.0,115147496.0,"Petroleum oils, refined"
3,2012,4,31,38,1076,0.0,383112.0,Anti-knock
4,2012,4,31,84,1625,0.0,50700.0,Equipment for temperature change of materials


Check what country ID corresponds to Russia

In [5]:
russia_id = countries[countries.country == "Russian Federation"].country_id.values[0]
ukraine_id = countries[countries.country == "Ukraine"].country_id.values[0]

# New Approach
So I'm thinking of a different plot. First, you select all countries that import a lot (threshold of your choice) of a product p from Russia (and/or Ukraine). Then you divide them into two groups: the first got a low risk score from GAT-total in 2021 (and/or 2013), the other got a high score. On the y-axis you can then report the % of those countries who were actually affected (according to our definition of being affected) in 2022 (and/or 2014, thank god Russia always invades at the beginning of the calendar year).

Ideally the bar with high risk scores should be higher than the one with low risk scores.

Select a base year, country to evaluate, and product

In [31]:
base_year = 2013

country = "Russia" # or Ukraine
country_id = russia_id if country == "Russia" else ukraine_id

code = "31"


print(f"Evaluating country: {country} ({country_id}) | commodity: {code} ({products[products.code == code].name_short_en.values[0]})")

Evaluating country: Russia (643) | commodity: 31 (Fertilisers)


Calculate main exports from country, given by SRCA greater than 1

In [ ]:
# Product 99 is undefined
if "XX" in rca_dict[base_year].columns:
    rca_dict[base_year] = rca_dict[base_year].drop("XX", axis=1)
if "99" in rca_dict[base_year].columns:
    rca_dict[base_year] = rca_dict[base_year].drop("99", axis=1)

high_scra = rca_dict[base_year].loc[russia_id][rca_dict[base_year].loc[russia_id] > 1]

pd.DataFrame(high_scra.sort_values(ascending=False)[:5]).reset_index()

KeyError: "['XX'] not found in axis"

Look at the median and mean export

In [21]:
export_median = atlas_data.loc[(atlas_data.country_id == russia_id) & 
                               (atlas_data.code == code) & (atlas_data.year == base_year) & 
                               (atlas_data.export_value > 0)].export_value.median()

export_mean = atlas_data.loc[(atlas_data.country_id == russia_id) & 
                              (atlas_data.code == code) & (atlas_data.year == base_year) & 
                              (atlas_data.export_value > 0)].export_value.mean()

print(f"Median export value of product {code} from Russia in {base_year}: {export_median:,.2f}")
print(f"Mean export value of product {code} from Russia in {base_year}: {export_mean:,.2f}")

Median export value of product 75 from Russia in 2013: 971,171.50
Mean export value of product 75 from Russia in 2013: 70,293,678.25


Select either median or mean as the threshold to consider an "important" importer of X product

In [22]:
method = "median"  # "median" or "mean"
threshold = export_median if method == "median" else export_mean

high_importers_of_x_prod = atlas_data.loc[(atlas_data.country_id == russia_id) & 
                                          (atlas_data.code == code) & (atlas_data.year == base_year) & 
                                          (atlas_data.export_value > threshold)].sort_values(by="export_value", ascending=False)

In [23]:
print(f'Number of importers of product {code} from Russia in {base_year}: {len(atlas_data.loc[(atlas_data.country_id == russia_id) & (atlas_data.code == code) & (atlas_data.year == base_year) &  (atlas_data.export_value > 0)])}')
print(f'Number of high importers of product {code} from Russia in {base_year}: {len(high_importers_of_x_prod)}')

Number of importers of product 75 from Russia in 2013: 60
Number of high importers of product 75 from Russia in 2013: 30


In [24]:
high_importers_of_x_prod.head(2)

,year,country_id,partner_country_id,code,product_id,export_value,import_value,name_short_en
1588615,2013,643,528,75,9203,1.697159e+09,4168974.0,Nickel tubes and pipesOther articles of nickel...
1584394,2013,643,156,75,9206,1.059858e+09,2753170.0,Nickel platesNickel tubes and pipesOther artic...


In [25]:
risk.head(2)

,year,model_type,graph_type,code,y,preds,probs,country_id,import_value,risk_score
0,2012,GAT,export,01,0,0,1.580000e-07,533,4950261.0,0.784239
1,2012,GAT,export,01,0,0,3.740000e-08,4,10045869.0,0.375509


In [26]:
high_importers_of_x_prod = high_importers_of_x_prod.merge(risk, 
                                                          left_on=["year", "code", "partner_country_id"], 
                                                          right_on=["year", "code", "country_id"])

In [27]:
risk_threshold = get_best_threshold(graph=graph, model="GAT")

high_importers_of_x_prod_high_risk = high_importers_of_x_prod[high_importers_of_x_prod.probs >= risk_threshold]
high_importers_of_x_prod_low_risk = high_importers_of_x_prod[high_importers_of_x_prod.probs < risk_threshold]

In [28]:
print(f"High Risk: {len(high_importers_of_x_prod_high_risk)}")
print(f"Low Risk: {len(high_importers_of_x_prod_low_risk)}")

High Risk: 6
Low Risk: 24


In [30]:
print(len(high_importers_of_x_prod_high_risk[high_importers_of_x_prod_high_risk.y == 1]), high_importers_of_x_prod_high_risk.y.mean())
print(len(high_importers_of_x_prod_low_risk[high_importers_of_x_prod_low_risk.y == 1]), high_importers_of_x_prod_low_risk.y.mean())

0 0.0
0 0.0
